# EDA 003.01 — What Is Feature Engineering

**Kaggle Feature Engineering Course — Lesson 1**
**Reference:** [What Is Feature Engineering](https://www.kaggle.com/code/ryanholbrook/what-is-feature-engineering) by Ryan Holbrook

---

## Learning Objectives

By the end of this notebook you will understand:

1. What feature engineering is and **why it matters**
2. The **goal** of feature engineering — making data easier for a model to learn from
3. The difference between **raw features** and **engineered features**
4. The main **types of transformations** applied to features
5. How feature quality directly impacts **model performance**

---

## What Is Feature Engineering?

> **Feature engineering** is the process of using domain knowledge to create, transform, or select input variables (features) that make machine learning algorithms work better.

A model can only learn what you give it. If the signal is buried in a raw form that's hard for the model to extract, no amount of hyperparameter tuning will compensate.

**Analogy:** A recipe analogy — raw ingredients (data) can be presented in ways that make cooking (learning) easier. Chopped garlic releases more flavour than whole cloves. Similarly, `log(price)` may expose patterns in price data that raw `price` hides.

---

## The Feature Engineering Workflow

```
Raw Data
    │
    ▼
Understand the domain (what does each column mean?)
    │
    ▼
Identify useful signals (which columns relate to target?)
    │
    ▼
Create / Transform features
    │
    ▼
Evaluate impact (does the new feature improve model score?)
    │
    ▼
Iterate
```

---

## Types of Feature Engineering

| Type | Example | Technique |
|---|---|---|
| **Mathematical transforms** | `log(price)`, `sqrt(area)` | Reduce skew, stabilise variance |
| **Interaction features** | `length × width = area` | Capture combined effect |
| **Aggregation features** | Mean price per neighbourhood | Encode group statistics |
| **Encoding categoricals** | `city → frequency` | Make categories learnable |
| **Date/time decomposition** | `hour`, `day_of_week`, `is_weekend` | Extract temporal patterns |
| **Binning / discretisation** | `age → [child, teen, adult, senior]` | Group continuous values |
| **Text features** | word count, TF-IDF | Extract signal from free text |
| **Dimensionality reduction** | PCA components | Compress correlated features |

---

## Popular Datasets to Practise Feature Engineering

| Dataset | Link | Why it's good |
|---|---|---|
| House Prices (Ames) | [Kaggle](https://www.kaggle.com/c/house-prices-advanced-regression-techniques) | 79 features, regression, lots of scope for transforms |
| Titanic | [Kaggle](https://www.kaggle.com/c/titanic) | Classification, mixed types, missing values, name parsing |
| NYC Taxi Trips | [Kaggle](https://www.kaggle.com/c/nyc-taxi-trip-duration) | Datetime, geospatial, distance calculation |
| Credit Card Fraud | [Kaggle](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud) | PCA-compressed, imbalanced, anomaly detection |
| Instacart Orders | [Kaggle](https://www.kaggle.com/c/instacart-market-basket-analysis) | Aggregation features, user behaviour |
| Ames Housing | [UCI](http://jse.amstat.org/v19n3/decock.pdf) | Detailed, requires domain knowledge |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.model_selection import cross_val_score

sns.set_theme(style="whitegrid")
%matplotlib inline

---
## Demo 1 — Why Raw Features Are Not Always Enough

We'll generate a dataset where the true relationship between `x` and `y` is non-linear (`y = x²`).  
A linear model on raw `x` will fail — but after feature engineering (`x²`), it succeeds perfectly.

In [ ]:
np.random.seed(42)
x = np.linspace(-3, 3, 200)
y = x**2 + np.random.normal(0, 0.3, 200)  # true signal: y = x^2

df = pd.DataFrame({'x': x, 'y': y})

# --- Model 1: Raw feature ---
X_raw = df[['x']]
model_raw = LinearRegression().fit(X_raw, y)
r2_raw = r2_score(y, model_raw.predict(X_raw))

# --- Model 2: Engineered feature (x^2) ---
df['x_squared'] = df['x'] ** 2
X_eng = df[['x_squared']]
model_eng = LinearRegression().fit(X_eng, y)
r2_eng = r2_score(y, model_eng.predict(X_eng))

print(f"R² using raw feature (x)  : {r2_raw:.4f}")
print(f"R² using engineered (x²)  : {r2_eng:.4f}")

# --- Plot ---
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].scatter(x, y, alpha=0.4, s=15)
axes[0].plot(np.sort(x), model_raw.predict(np.sort(x).reshape(-1,1)), color='red', linewidth=2)
axes[0].set_title(f'Raw feature: x  →  R² = {r2_raw:.3f}', fontsize=12)
axes[0].set_xlabel('x'); axes[0].set_ylabel('y')

axes[1].scatter(df['x_squared'], y, alpha=0.4, s=15, color='orange')
xs2 = np.sort(df['x_squared'].values)
axes[1].plot(xs2, model_eng.predict(xs2.reshape(-1,1)), color='red', linewidth=2)
axes[1].set_title(f'Engineered feature: x²  →  R² = {r2_eng:.3f}', fontsize=12)
axes[1].set_xlabel('x²'); axes[1].set_ylabel('y')

plt.suptitle('Feature Engineering: Exposing the True Signal', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## Demo 2 — Feature Representation Matters

Here we show how **three different representations of the same data** produce very different model scores.  
The data: house `area` (m²) and house `price`.

- **Raw price** — heavily right-skewed
- **log(price)** — approximately normal → better fit for linear regression
- **sqrt(price)** — moderate compression

In [ ]:
np.random.seed(7)
area = np.random.uniform(40, 300, 300)
# True: price = k * area^1.4 (non-linear scaling)
price = 500 * area**1.4 * np.exp(np.random.normal(0, 0.15, 300))

df = pd.DataFrame({'area': area, 'price': price})
df['log_price'] = np.log(price)
df['sqrt_price'] = np.sqrt(price)

# Plot distributions
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col, color in zip(axes, ['price', 'log_price', 'sqrt_price'],
                           ['steelblue', 'darkorange', 'seagreen']):
    ax.hist(df[col], bins=30, color=color, edgecolor='white', alpha=0.85)
    ax.set_title(col, fontsize=12)
    skew = df[col].skew()
    ax.set_xlabel(f'skewness = {skew:.2f}')

plt.suptitle('Distributions of Different Price Representations', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

# R² comparison
X = df[['area']]
for target in ['price', 'log_price', 'sqrt_price']:
    r2 = r2_score(df[target], LinearRegression().fit(X, df[target]).predict(X))
    print(f"R² predicting {target:12s} from area: {r2:.4f}")

---
## Key Takeaways

- Feature engineering **bridges the gap** between raw data and what a model can learn
- The **same data** presented differently can dramatically change model performance
- Good features often come from **domain knowledge** — understanding what the data represents
- Feature engineering is **iterative** — create, evaluate, refine
- More data and better features usually outperform more complex models on the same data

---

## Further Reading

- [Feature Engineering for Machine Learning](https://www.oreilly.com/library/view/feature-engineering-for/9781491953235/) — Alice Zheng & Amanda Casari (O'Reilly)
- [Kaggle Feature Engineering Course](https://www.kaggle.com/learn/feature-engineering) — Ryan Holbrook
- [Hands-On Machine Learning, Ch. 2](https://www.oreilly.com/library/view/hands-on-machine-learning/9781492032632/) — Aurélien Géron
- [Feature Engineering and Selection](http://www.feat.engineering/) — Kuhn & Johnson (free online)